# 03 — PhysioNet-shaped protocol occurrences

External protocol labels, button boundaries, self-reports, and native-rate
wearable measurements must remain distinct. FeatureGraph does not infer stress.


In [ ]:
import numpy as np
import pandas as pd
import featuregraph as fg

states = ["baseline", "baseline", "task", "task",
          "task", "rest", "rest", "unassigned"]
rows = []
for subject, offset in [("S01", 0.0), ("S02", 2.0)]:
    for second, state in enumerate(states):
        rows.append({
            "subject_id": subject, "second": second, "protocol_state": state,
            "hr": 68 + offset + [0, 1, 7, 11, 9, 5, 2, 1][second],
            "eda": .20 + offset / 100 + [0, .01, .08, .12, .10, .05, .02, .01][second],
            "temp": 32 + offset / 10 + [0, 0, .1, .1, .2, .2, .1, .1][second],
        })
observations = pd.DataFrame(rows)


In [ ]:
contract = {
    "version": "state-contract-v1",
    "state_column": "protocol_state",
    "group_by": "subject_id",
    "events": {"enter_protocol_state": {"type": "enter_label"},
               "exit_protocol_state": {"type": "exit_label"}},
}
compiled = fg.compile_states(observations, contract)
compiled.observations.head(10)


FeatureGraph supplies occurrence identity and boundaries.
Pandas supplies measurements requested inside the declared intervals.
`fg.transition.Transition` is not used to redetect external protocol labels.


In [ ]:
objects = compiled.observations.groupby(
    ["subject_id", "state_occurrence_id", "state"], sort=False
).agg(
    start_second=("second", "min"), end_second=("second", "max"),
    sample_count=("second", "size"), hr_median=("hr", "median"),
    eda_median=("eda", "median"), temp_median=("temp", "median"),
).reset_index().rename(columns={"state": "protocol_state"})

reports = pd.DataFrame({
    "subject_id": ["S01", "S02"], "protocol_state": ["task", "task"],
    "self_reported_stress": [6, 7],
})
objects = objects.merge(
    reports, on=["subject_id", "protocol_state"],
    how="left", validate="many_to_one",
)
objects


In [ ]:
subject = observations.query("subject_id == 'S01'").reset_index(drop=True)
occurrences = fg.from_state_sequence(
    subject["protocol_state"], signal=subject["hr"], times=subject["second"],
    group_id="S01", dataset="physionet-shaped-protocol-tutorial",
    signal_name="heart_rate", signal_unit="beats/minute",
    detector="published_protocol", software_version=fg.__version__,
)
occurrences.object_table()


In [ ]:
assert len(objects) == 8
assert objects.groupby("subject_id").size().eq(4).all()
assert compiled.observations.groupby("subject_id")[
    "state_occurrence_id"
].min().eq(0).all()
assert objects.loc[
    objects["protocol_state"].eq("task"), "self_reported_stress"
].notna().all()
assert np.array_equal(
    occurrences.reconstruct_states(), subject["protocol_state"].to_numpy()
)


These are descriptive protocol measurements, not a stress
detector or biomarker validation. The maintained study scales the pattern to
33 participants and 248 declared occurrences across two protocol versions.
